# Lab 04 — Bronze ingestion

This notebook loads the prepared Online Retail Parquet batch into the pre-created Bronze Delta table.

Bronze preserves the source values—including duplicates and questionable business values—while adding technical metadata for lineage, auditing, and safe replay.

## Objectives

- load either the `initial` or `incremental` staged batch;
- retain every source column without Silver-layer cleaning;
- add stable record identifiers, batch metadata, and file metadata;
- use an insert-only Delta `MERGE` so rerunning the same batch is idempotent;
- verify row counts, lineage, uniqueness, and Delta history.

> **Structure boundary:** `lab04_00_setup` creates the Bronze table outside the production Job.
>
> **Layer boundary:** business cleaning, deduplication, and quarantine rules belong in `lab04_03_silver_quality`, not here.


## 1. Load shared configuration

The configuration notebook supplies catalog, schema, volume paths, table names, contract version, batch ID, and reset controls. Keep all Lab 4 notebooks beside one another so the relative `%run` path remains valid.

In [0]:
%run ./lab04_00_config

# Lab 04 — Runtime Configuration

This notebook is intentionally **DDL-free**.

It:
- defines runtime widgets and validates their values;
- builds reusable paths and table names;
- loads the version-controlled YAML data contracts;
- exposes one runtime-selected contract plus the v1/v2 references required by the schema-governance demonstrations.

It does **not** create catalogs, schemas, volumes, folders, or tables.

Run `lab04_00_setup` manually once when the Lab 4 structure must be created or verified. Production Job tasks may `%run ./lab04_00_config` safely.


runtime_selection,contract_name,yaml_version,governance_status,column_count,supersedes,runtime_selected
v1,online_retail,1,active,8,null,true


Runtime configuration ready: dbr_dev.parvinbadalov
Volume: /Volumes/dbr_dev/parvinbadalov/lab04_silver_quality
Source workbook: /Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/source/Online Retail.xlsx
Runtime contract: online_retail v1 (governance status: active)
Schema policy: fail


In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

print(f"Bronze target: {bronze_table}")
print(f"Requested batch: {batch_id}")
print(f"Contract version: {contract_version}")

Bronze target: dbr_dev.parvinbadalov.lab04_bronze_retail
Requested batch: initial
Contract version: v1


## 2. Select and verify the staged batch

`batch_id=initial` reads the large first delivery. `batch_id=incremental` reads the later delivery. Other staged datasets are reserved for the schema-evolution and data-quality notebooks, so this ingestion notebook rejects unsupported batch names.

In [0]:
batch_sources = {
    "initial": paths["staging_initial"],
    "incremental": paths["staging_incremental"],
}

if batch_id not in batch_sources:
    raise ValueError(
        f"Unsupported batch_id={batch_id!r}. "
        f"Choose one of {sorted(batch_sources)} for Bronze ingestion."
    )

bronze_source_path = batch_sources[batch_id]
source_files = [item for item in dbutils.fs.ls(bronze_source_path) if item.path.endswith(".parquet")]

if not source_files:
    raise FileNotFoundError(
        f"No Parquet files found under {bronze_source_path}. "
        "Run lab04_01_source_preparation.ipynb first."
    )

print(f"Source path: {bronze_source_path}")
print(f"Parquet files discovered: {len(source_files)}")

Source path: /Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/staging/initial
Parquet files discovered: 16


## 3. Read source values and file metadata

Databricks exposes a hidden `_metadata` struct for file-based reads. Capturing its fields in Bronze makes every record traceable to a specific file, file size, and modification time.

In [0]:
raw_batch_df = (
    spark.read
    .format("parquet")
    .load(bronze_source_path)
    .select(
        "*",
        F.col("_metadata.file_path").alias("_input_file_path"),
        F.col("_metadata.file_name").alias("_input_file_name"),
        F.col("_metadata.file_size").alias("_input_file_size"),
        F.col("_metadata.file_modification_time").alias("_input_file_modified_at"),
    )
)

required_prepared_columns = set(expected_source_columns) | {
    "_source_row_number",
    "_source_file",
    "_source_sheet",
    "_prepared_at_utc",
    "_record_hash",
}
missing_columns = sorted(required_prepared_columns - set(raw_batch_df.columns))
if missing_columns:
    raise AssertionError(f"Prepared batch is missing columns: {missing_columns}")

incoming_row_count = raw_batch_df.count()
incoming_business_duplicates = (
    incoming_row_count
    - raw_batch_df.select(*expected_source_columns).distinct().count()
)

print(f"Incoming rows: {incoming_row_count:,}")
print(f"Business duplicate rows retained in Bronze input: {incoming_business_duplicates:,}")
raw_batch_df.printSchema()

Incoming rows: 433,737
Business duplicate rows retained in Bronze input: 3,427
root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: long (nullable = true)
 |-- InvoiceDate: timestamp_ntz (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- _source_row_number: long (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _source_sheet: string (nullable = true)
 |-- _prepared_at_utc: timestamp (nullable = true)
 |-- _record_hash: string (nullable = true)
 |-- _input_file_path: string (nullable = false)
 |-- _input_file_name: string (nullable = false)
 |-- _input_file_size: long (nullable = false)
 |-- _input_file_modified_at: timestamp (nullable = false)



## 4. Add Bronze technical metadata

The stable `_bronze_record_id` combines workbook, sheet, and original worksheet row number. Unlike a hash of business values, it distinguishes two genuinely separate source rows even when their business fields are identical. That lets Bronze preserve source duplicates while an insert-only `MERGE` prevents duplicates caused by rerunning this notebook.

In [0]:
bronze_batch_df = (
    raw_batch_df
    .withColumn(
        "_bronze_record_id",
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(F.col("_source_file"), F.lit("<NULL>")),
                F.coalesce(F.col("_source_sheet"), F.lit("<NULL>")),
                F.col("_source_row_number").cast("string"),
            ),
            256,
        ),
    )
    .withColumn("_batch_id", F.lit(batch_id))
    .withColumn("_source_system", F.lit("uci_online_retail"))
    .withColumn("_contract_version", F.lit(contract_version))
    .withColumn("_bronze_ingested_at", F.current_timestamp())
    .withColumn("_bronze_ingestion_date", F.current_date())
)

source_key_quality = bronze_batch_df.agg(
    F.count("*").alias("rows"),
    F.countDistinct("_bronze_record_id").alias("distinct_ids"),
    F.sum(F.col("_bronze_record_id").isNull().cast("int")).alias("null_ids"),
).first()

if source_key_quality["rows"] != source_key_quality["distinct_ids"]:
    raise AssertionError("The staged batch contains duplicate technical source-row identities.")
if source_key_quality["null_ids"] != 0:
    raise AssertionError("The staged batch contains null Bronze record IDs.")

display(bronze_batch_df.limit(20))

InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,_source_row_number,_source_file,_source_sheet,_prepared_at_utc,_record_hash,_input_file_path,_input_file_name,_input_file_size,_input_file_modified_at,_bronze_record_id,_batch_id,_source_system,_contract_version,_bronze_ingested_at,_bronze_ingestion_date
536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01T08:26:00.000,7.65,17850,United Kingdom,7,Online Retail.xlsx,Online Retail,2026-08-10T02:04:15.872Z,0d605467e0138f6396f9c149ce5ad5ab7096c7056518578056ef5dc2dc38dfbd,dbfs:/Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/staging/initial/part-00005-tid-6964098672136194727-09e5ab4c-18ba-4f14-84f4-484371274d7a-182-1.c000.snappy.parquet,part-00005-tid-6964098672136194727-09e5ab4c-18ba-4f14-84f4-484371274d7a-182-1.c000.snappy.parquet,2381524,2026-08-10T02:04:18.000Z,7d07613aeedc44e44ab851ad437283d55529b57cf01ca858f2db461bb85de370,initial,uci_online_retail,v1,2026-08-10T20:51:44.605Z,2026-08-10
536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,2010-12-01T08:34:00.000,1.69,13047,United Kingdom,11,Online Retail.xlsx,Online Retail,2026-08-10T02:04:15.872Z,75454c4ed0bc3e1e48f278af66b0f1de1a082a6ed8765e348cd8bc5f593895c6,dbfs:/Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/staging/initial/part-00005-tid-6964098672136194727-09e5ab4c-18ba-4f14-84f4-484371274d7a-182-1.c000.snappy.parquet,part-00005-tid-6964098672136194727-09e5ab4c-18ba-4f14-84f4-484371274d7a-182-1.c000.snappy.parquet,2381524,2026-08-10T02:04:18.000Z,0d5fdb0a4b2edfc4ee3259f49e7bd6c4b46b8c761a80834678c5cf3633f1459a,initial,uci_online_retail,v1,2026-08-10T20:51:44.605Z,2026-08-10
536370,22631,CIRCUS PARADE LUNCH BOX,24,2010-12-01T08:45:00.000,1.95,12583,France,39,Online Retail.xlsx,Online Retail,2026-08-10T02:04:15.872Z,de898c3c66d559851b5857be3e204f5f314afca2494d150bd328f458b6554217,dbfs:/Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/staging/initial/part-00005-tid-6964098672136194727-09e5ab4c-18ba-4f14-84f4-484371274d7a-182-1.c000.snappy.parquet,part-00005-tid-6964098672136194727-09e5ab4c-18ba-4f14-84f4-484371274d7a-182-1.c000.snappy.parquet,2381524,2026-08-10T02:04:18.000Z,327dec9292ed7737e1a8d52e7516c001ea91b1a11f599217607074decf10fbc1,initial,uci_online_retail,v1,2026-08-10T20:51:44.605Z,2026-08-10
536373,71053,WHITE METAL LANTERN,6,2010-12-01T09:02:00.000,3.39,17850,United Kingdom,52,Online Retail.xlsx,Online Retail,2026-08-10T02:04:15.872Z,0abcd2d9bfc771dff355e7b4094672acbbe86fa8d32995a8e050038d83029270,dbfs:/Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/staging/initial/part-00005-tid-6964098672136194727-09e5ab4c-18ba-4f14-84f4-484371274d7a-182-1.c000.snappy.parquet,part-00005-tid-6964098672136194727-09e5ab4c-18ba-4f14-84f4-484371274d7a-182-1.c000.snappy.parquet,2381524,2026-08-10T02:04:18.000Z,3b1e5efd95b0a8634669b33f216781f3edc868c20b8c8451882a25bb0d684385,initial,uci_online_retail,v1,2026-08-10T20:51:44.605Z,2026-08-10
536375,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01T09:32:00.000,2.75,17850,United Kingdom,70,Online Retail.xlsx,Online Retail,2026-08-10T02:04:15.872Z,0a8f0365b08b7843d372e7b3d2401c15a4a73e3de85cb945722010a7b1c66fad,dbfs:/Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/staging/initial/part-00005-tid-6964098672136194727-09e5ab4c-18ba-4f14-84f4-484371274d7a-182-1.c000.snappy.parquet,part-00005-tid-6964098672136194727-09e5ab4c-18ba-4f14-84f4-484371274d7a-182-1.c000.snappy.parquet,2381524,2026-08-10T02:04:18.000Z,0da6b97db9597b7d5e64a5d083688bc27cdd93f7ab330fa7065523293f3a70c9,initial,uci_online_retail,v1,2026-08-10T20:51:44.605Z,2026-08-10
536381,15056BL,EDWARDIAN PARASOL BLACK,2,2010-12-01T09:41:00.000,5.95,15311,United Kingdom,134,Online Retail.xlsx,Online Retail,2026-08-10T02:04:15.872Z,d5b03a37038450c09565d5cda7b2bf93cb9639be8fb889805d7f3493b0c5ca40,dbfs:/Volumes/dbr_dev/parvinbadalov/lab04_silver_quality/staging/initial/part-00005-tid-6964098672136194727-09e5ab4c-18ba-4f14-84f4-484371274d7a-182-1.c000.snappy.parquet,part-00005-tid-6964098672136194

## 5. Verify the pre-created Bronze Delta table

The Bronze table structure is created by **`lab04_00_setup`**, which is run manually and is not part of the production Job.

This notebook performs no catalog/schema/table DDL. It only validates that the required Bronze target exists and matches the incoming technical schema.


In [0]:
if reset_demo_objects:
    raise ValueError(
        "reset_demo_objects=true is not allowed inside the production Job. "
        "Run lab04_00_setup manually when a clean structural rebuild is required."
    )

if not spark.catalog.tableExists(bronze_table):
    raise RuntimeError(
        f"Required Bronze target does not exist: {bronze_table}. "
        "Run lab04_00_setup manually before executing the Job."
    )

target_schema = {
    field.name: field.dataType.simpleString()
    for field in spark.table(bronze_table).schema.fields
}

incoming_schema = {
    field.name: field.dataType.simpleString()
    for field in bronze_batch_df.schema.fields
}

missing_in_target = sorted(set(incoming_schema) - set(target_schema))
extra_in_target = sorted(set(target_schema) - set(incoming_schema))
type_mismatches = {
    name: {
        "incoming": incoming_schema[name],
        "target": target_schema[name],
    }
    for name in sorted(set(incoming_schema) & set(target_schema))
    if incoming_schema[name] != target_schema[name]
}

if missing_in_target or extra_in_target or type_mismatches:
    raise AssertionError(
        "Bronze target schema does not match the prepared batch. "
        f"Missing in target: {missing_in_target}; "
        f"extra in target: {extra_in_target}; "
        f"type mismatches: {type_mismatches}. "
        "Run lab04_00_setup manually if the structural contract changed."
    )

print(f"✅ Pre-created Bronze target is ready: {bronze_table}")


✅ Pre-created Bronze target is ready: dbr_dev.parvinbadalov.lab04_bronze_retail


## 6. Insert only unseen source rows with MERGE

This is an insert-only `MERGE`: a row is inserted only when its stable source-row identity is not already present. Existing Bronze rows are never updated because Bronze is an immutable record of what arrived.

In [0]:
bronze_delta = DeltaTable.forName(spark, bronze_table)
before_merge_count = spark.table(bronze_table).count()

(
    bronze_delta.alias("target")
    .merge(
        bronze_batch_df.alias("source"),
        "target._bronze_record_id = source._bronze_record_id",
    )
    .whenNotMatchedInsertAll()
    .execute()
)

after_merge_count = spark.table(bronze_table).count()
inserted_rows = after_merge_count - before_merge_count

print(f"Rows before MERGE: {before_merge_count:,}")
print(f"Rows inserted: {inserted_rows:,}")
print(f"Rows after MERGE: {after_merge_count:,}")

Rows before MERGE: 433,737
Rows inserted: 0
Rows after MERGE: 433,737


## 7. Prove idempotency by replaying the same batch

A production pipeline must tolerate retries. The same `MERGE` is executed a second time and the table count must remain unchanged. This is the key evidence that a job retry will not duplicate the batch.

In [0]:
count_before_replay = spark.table(bronze_table).count()

(
    DeltaTable.forName(spark, bronze_table).alias("target")
    .merge(
        bronze_batch_df.alias("source"),
        "target._bronze_record_id = source._bronze_record_id",
    )
    .whenNotMatchedInsertAll()
    .execute()
)

count_after_replay = spark.table(bronze_table).count()
if count_after_replay != count_before_replay:
    raise AssertionError(
        f"Idempotency failed: count changed from {count_before_replay} "
        f"to {count_after_replay} during replay."
    )

print("✅ Idempotency check passed: replay inserted 0 rows.")
print(f"Bronze row count remains {count_after_replay:,}.")

✅ Idempotency check passed: replay inserted 0 rows.
Bronze row count remains 433,737.


## 8. Validate Bronze quality and lineage

Bronze quality checks are technical rather than business-facing: IDs and file lineage must be complete, and the technical ID must be unique. Exact business duplicates are intentionally retained so Silver can handle them explicitly and transparently.

In [0]:
bronze_df = spark.table(bronze_table)
bronze_count = bronze_df.count()
business_distinct_count = bronze_df.select(*expected_source_columns).distinct().count()

technical_quality_df = bronze_df.agg(
    F.count("*").alias("bronze_rows"),
    F.countDistinct("_bronze_record_id").alias("distinct_bronze_ids"),
    F.sum(F.col("_bronze_record_id").isNull().cast("int")).alias("null_bronze_ids"),
    F.sum(F.col("_input_file_path").isNull().cast("int")).alias("null_file_paths"),
    F.countDistinct("_input_file_path").alias("processed_files"),
    F.countDistinct("_batch_id").alias("batch_count"),
)
quality = technical_quality_df.first().asDict()

if quality["bronze_rows"] != quality["distinct_bronze_ids"]:
    raise AssertionError("Bronze technical IDs are not unique.")
if quality["null_bronze_ids"] != 0 or quality["null_file_paths"] != 0:
    raise AssertionError(f"Bronze technical metadata is incomplete: {quality}")

print(f"Business duplicates retained for Silver: {bronze_count - business_distinct_count:,}")
display(technical_quality_df)
display(
    bronze_df
    .orderBy(F.col("_source_row_number"))
    .select(
        *expected_source_columns,
        "_bronze_record_id",
        "_batch_id",
        "_input_file_name",
        "_bronze_ingested_at",
    )
    .limit(20)
)

Business duplicates retained for Silver: 3,427


bronze_rows,distinct_bronze_ids,null_bronze_ids,null_file_paths,processed_files,batch_count
433737,433737,0,0,16,1


InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,_bronze_record_id,_batch_id,_input_file_name,_bronze_ingested_at
536365,71053,WHITE METAL LANTERN,6,2010-12-01T08:26:00.000,3.39,17850,United Kingdom,17000e568d97768ddfb50b30790fb358289f6cd675f7925ce9ada92655c8f70b,initial,part-00006-tid-762981857188285749-d78f4527-cbdb-4699-a0a9-42e14bfbe4f0-315-1.c000.snappy.parquet,2026-08-09T20:37:30.749Z
536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01T08:26:00.000,2.75,17850,United Kingdom,ebb5304c4d44f54ef8cb0c618c67b7c0439fdcce9f5c05922ec85d1d4bdbb90c,initial,part-00009-tid-762981857188285749-d78f4527-cbdb-4699-a0a9-42e14bfbe4f0-318-1.c000.snappy.parquet,2026-08-09T20:37:30.749Z
536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01T08:26:00.000,3.39,17850,United Kingdom,86a800eb6412b8679412adfaebea76891efe154342eaa161ce79ecb41cc1a55a,initial,part-00008-tid-762981857188285749-d78f4527-cbdb-4699-a0a9-42e14bfbe4f0-317-1.c000.snappy.parquet,2026-08-09T20:37:30.749Z
536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01T08:26:00.000,3.39,17850,United Kingdom,f871cfdcba9b6cfe9fa0c049fc80f84ac65f914f8848efe4af499c6fdac859cb,initial,part-00007-tid-762981857188285749-d78f4527-cbdb-4699-a0a9-42e14bfbe4f0-316-1.c000.snappy.parquet,2026-08-09T20:37:30.749Z
536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01T08:26:00.000,7.65,17850,United Kingdom,7d07613aeedc44e44ab851ad437283d55529b57cf01ca858f2db461bb85de370,initial,part-00005-tid-762981857188285749-d78f4527-cbdb-4699-a0a9-42e14bfbe4f0-314-1.c000.snappy.parquet,2026-08-09T20:37:30.749Z
536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01T08:26:00.000,4.25,17850,United Kingdom,6c4de07354ac2a12840cb7ab18e29b9d0299b88486a9bd6288b25d7347f167f2,initial,part-00004-tid-762981857188285749-d78f4527-cbdb-4699-a0a9-42e14bfbe4f0-313-1.c000.snappy.parquet,2026-08-09T20:37:30.749Z
536366,22633,HAND WARMER UNION JACK,6,2010-12-01T08:28:00.000,1.85,17850,United Kingdom,6c30ac1498c5debdf9d0ed0ba108ded5e8c27d4ddb9fa285b84e83b0125dd685,initial,part-00011-tid-762981857188285749-d78f4527-cbdb-4699-a0a9-42e14bfbe4f0-309-1.c000.snappy.parquet,2026-08-09T20:37:30.749Z
536366,22632,HAND WARMER RED POLKA DOT,6,2010-12-01T08:28:00.000,1.85,17850,United Kingdom,28fe3ea9b4ccfebcde298f07fc2aa7b5f42aa534599720dd4bd24e3934a6b02e,initial,part-00010-tid-762981857188285749-d78f4527-cbdb-4699-a0a9-42e14bfbe4f0-319-1.c000.snappy.parquet,2026-08-09T20:37:30.749Z
536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,2010-12-01T08:34:00.000,1.69,13047,United Kingdom,0d5fdb0a4b2edfc4ee3259f49e7bd6c4b46b8c761a80834678c5cf3633f1459a,initial,part-00005-tid-762981857188285749-d78f4527-cbdb-4699-a0a9-42e14bfbe4f0-314-1.c000.snappy.parquet,2026-08-09T20:37:30.749Z
536367,22745,POPPY'S PLAYHOUSE BEDROOM,6,2010-12-01T08:34:00.000,2.1,13047,United Kingdom,02125f4022794e37a34cf6850050bd9f6e83ea21125406cf82edc2b55e6d9509,initial,part-00001-tid-762981857188285749-d78f4527-cbdb-4699-a0a9-42e14bfbe4f0-311-1.c000.snappy.parquet,2026-08-09T20:37:30.749Z


## 9. Inspect Delta transaction history

Delta history records each table operation. The two MERGE entries provide audit evidence for the original ingestion and the zero-row replay used in the idempotency test.

In [0]:
history_df = spark.sql(f"DESCRIBE HISTORY {bronze_table}")
display(
    history_df.select(
        "version",
        "timestamp",
        "operation",
        "operationParameters",
        "operationMetrics",
    ).orderBy(F.col("version").desc())
)

version,timestamp,operation,operationParameters,operationMetrics
9,2026-08-10T20:51:53.000Z,MERGE,"Map(predicate -> [""(_bronze_record_id#35449 = _bronze_record_id#34488)""], clusterBy -> [], matchedPredicates -> [], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])","Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 0, numTargetBytesAdded -> 0, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 1792, materializeSourceTimeMs -> 9, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 0, numTargetRowsUpdated -> 0, numOutputRows -> 0, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 433737, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1744)"
8,2026-08-10T20:51:50.000Z,MERGE,"Map(predicate -> [""(_bronze_record_id#34683 = _bronze_record_id#34488)""], clusterBy -> [], matchedPredicates -> [], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])","Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 0, numTargetBytesAdded -> 0, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 1971, materializeSourceTimeMs -> 8, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 0, numTargetRowsUpdated -> 0, numOutputRows -> 0, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 433737, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1926)"
7,2026-08-10T15:33:27.000Z,MERGE,"Map(predicate -> [""(_bronze_record_id#12557 = _bronze_record_id#11538)""], clusterBy -> [], matchedPredicates -> [], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])","Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 0, numTargetBytesAdded -> 0, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 2261, materializeSourceTimeMs -> 13, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 0, numTargetRowsUpdated -> 0, numOutputRows -> 0, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 433737, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2209)"
6,2026-08-10T15:33:22.000Z,MERGE,"Map(predicate -> [""(_bronze_record_id#11791 = _bronze_record_id#11538)""], clusterBy -> [], matchedPredicates -> [], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])","Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 0, numTargetBytesAdded -> 0, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 7053, materializeSourceTimeMs -> 21, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 0, numTargetRowsUpdated -> 0, numOutputRows -> 0, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 433737, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 6923)"
5,2026-08-10T02:05:19.000Z,MERGE,"Map(predicate -> [""(_bronze_record_id#13213 = _bronze_record_id#12241)""], clusterBy -> [], matchedPredicates -> [], statsOnLoad -> false, notMatchedBySourcePredicates -> [], not

## 10. Completion checklist and next notebook

This notebook is complete when:

- the selected staged batch exists and its prepared schema is valid;
- the Bronze Delta table contains the expected new source rows;
- technical IDs are unique and file lineage is populated;
- the replay check reports zero inserted rows;
- Delta history shows the ingestion and replay MERGE operations.

Recommended evidence screenshots: the MERGE counts, idempotency result, technical-quality summary, and Delta history.

### Next notebook

Continue with **`lab04_03_silver_quality.ipynb`**. It will read this Bronze table, apply explicit data-quality rules, split valid and invalid rows, quarantine rejected records with reasons, and prepare the clean Silver candidate dataset.